In [1]:
from google.colab import files

uploaded = files.upload()

Saving event_log_ED_MMA_2026.csv to event_log_ED_MMA_2026.csv


Dataset Information

In [2]:
import pandas as pd

file_name = "event_log_ED_MMA_2026.csv"

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Dataset shape: (90965, 17)

Columns:
['visit_id', 'patient_id', 'initial_zone', 'age', 'month', 'day', 'gender', 'triage_code', 'triage_desc', 'disposition_code', 'disposition_desc', 'consult_desc', 'cdu_flag', 'consult_req_flag', 'consult_arrival_flag', 'event', 'timestamp']

First 5 records:


,visit_id,patient_id,initial_zone,age,month,day,gender,triage_code,triage_desc,disposition_code,disposition_desc,consult_desc,cdu_flag,consult_req_flag,consult_arrival_flag,event,timestamp
0,7448126,7236248,YZ,67,May,Wednesday,F,2.0,EMERGENCY,17,"Discharge to private home, condo, apt without ...",NaN,2,NaN,NaN,Ambulance Transfer,2021-05-12 01:16:00
1,7448126,7236248,YZ,67,May,Wednesday,F,2.0,EMERGENCY,17,"Discharge to private home, condo, apt without ...",NaN,2,NaN,NaN,Triage,2021-05-12 01:07:00
2,7448126,7236248,YZ,67,May,Wednesday,F,2.0,EMERGENCY,17,"Discharge to private home, condo, apt without ...",NaN,2,NaN,NaN,Left ED,2021-05-12 05:09:00
3,7448126,7236248,YZ,67,May,Wednesday,F,2.0,EMERGENCY,17,"Discharge to private home, condo, apt without ...",NaN,2,NaN,NaN,Registration,2021-05-12 01:18:00
4,7448126,7236248,YZ,67,May,Wednesday,F,2.0,EMERGENCY,17,"Discharge to private home, condo, apt without ...",NaN,2,NaN,NaN,Discharge,2021-05-12 04:56:00


Converting in to careflow schema

In [3]:

import pandas as pd
from pathlib import Path

# Normalize source column names.
df.columns = df.columns.str.strip().str.lower()

required_source_columns = [
    "visit_id",
    "event",
    "timestamp",
]

missing = [
    col for col in required_source_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Required source columns not found: {missing}. "
        f"Available columns: {df.columns.tolist()}"
    )

# Preserve the source dataset; work on a copy.
clean = df.copy()

# Map source columns to CareFlow names.
clean = clean.rename(columns={
    "visit_id": "case_id",
    "event": "activity",
})

# Parse timestamps and identify invalid records.
clean["timestamp"] = pd.to_datetime(
    clean["timestamp"],
    errors="coerce"
)

invalid_timestamp_count = clean["timestamp"].isna().sum()

# Exclude records that cannot be used in chronological
# process mining, and records without a case/activity.
clean = clean.dropna(
    subset=["case_id", "activity", "timestamp"]
).copy()

clean["case_id"] = clean["case_id"].astype(str)
clean["activity"] = clean["activity"].astype(str)

clean = clean.sort_values(
    ["case_id", "timestamp"],
    kind="stable"
).reset_index(drop=True)

# Derive event order within each visit.
clean["sequence"] = (
    clean.groupby("case_id").cumcount() + 1
)

# Keep core fields and useful source attributes.
preferred_columns = [
    "case_id",
    "activity",
    "timestamp",
    "sequence",
    "triage_code",
    "initial_zone",
    "disposition_code",
    "disposition_desc",
    "age",
    "gender",
]

output_columns = [
    col for col in preferred_columns
    if col in clean.columns
]

careflow_df = clean[output_columns].copy()

# Save the normalized dataset.
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "careflow_event_log.csv"

careflow_df.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Output:", output_file)
print("Rows:", len(careflow_df))
print("Unique cases:", careflow_df["case_id"].nunique())
print("Invalid timestamps excluded:", invalid_timestamp_count)
print("\nFinal columns:", careflow_df.columns.tolist())
display(careflow_df.head(10))


Output: data/processed/careflow_event_log.csv
Rows: 90965
Unique cases: 16011
Invalid timestamps excluded: 0

Final columns: ['case_id', 'activity', 'timestamp', 'sequence', 'triage_code', 'initial_zone', 'disposition_code', 'disposition_desc', 'age', 'gender']


,case_id,activity,timestamp,sequence,triage_code,initial_zone,disposition_code,disposition_desc,age,gender
0,5240985,Left ED,2021-04-28 00:00:00,1,2.0,NaN,63,Left After Triage,8,M
1,5240985,Discharge,2021-04-28 00:00:00,2,2.0,NaN,63,Left After Triage,8,M
2,5240985,Triage,2021-04-28 01:30:00,3,2.0,NaN,63,Left After Triage,8,M
3,5240985,Registration,2021-04-28 01:45:00,4,2.0,NaN,63,Left After Triage,8,M
4,7214779,Triage,2021-05-21 11:17:00,1,3.0,YZ,63,Left After Triage,27,M
5,7214779,Registration,2021-05-21 11:27:00,2,3.0,YZ,63,Left After Triage,27,M
6,7214779,Left ED,2021-05-21 12:09:00,3,3.0,YZ,63,Left After Triage,27,M
7,7214779,Discharge,2021-05-21 12:09:00,4,3.0,YZ,63,Left After Triage,27,M
8,7268604,Ambulance Arrival,2021-04-11 00:00:00,1,2.0,Resus,7,Admit to reporting facility as inpatient to an...,27,F
9,7268604,Triage,2021-04-11 13:04:00,2,2.0,Resus,7,Admit to reporting facility as inpatient to an...,27,F


Dataset Validation

In [4]:

# 1. Check required columns
required = ["case_id", "activity", "timestamp", "sequence"]
assert all(col in careflow_df.columns for col in required)

# 2. Check missing values in core fields
print("Missing values:")
print(careflow_df[required].isna().sum())

# 3. Check duplicate event keys
print(
    "Duplicate case_id + sequence:",
    careflow_df.duplicated(
        subset=["case_id", "sequence"]
    ).sum()
)

# 4. Check chronological order
time_differences = (
    careflow_df.groupby("case_id")["timestamp"]
    .diff()
)

print(
    "Negative time differences:",
    (time_differences < pd.Timedelta(0)).sum()
)

# 5. Summary
print("\n=== CAREFLOW DATA SUMMARY ===")
print("Total events:", len(careflow_df))
print("Unique cases:", careflow_df["case_id"].nunique())
print("Unique activities:", careflow_df["activity"].nunique())

print("\nTop activities:")
print(careflow_df["activity"].value_counts().head(15))

print("\nValidation complete.")


Missing values:
case_id      0
activity     0
timestamp    0
sequence     0
dtype: int64
Duplicate case_id + sequence: 0
Negative time differences: 0

=== CAREFLOW DATA SUMMARY ===
Total events: 90965
Unique cases: 16011
Unique activities: 9

Top activities:
activity
Discharge             16011
Registration          16011
Triage                16010
Assessment            15848
Left ED               15825
Ambulance Arrival      3115
Ambulance Transfer     3115
Consult Request        2718
Consult Arrival        2312
Name: count, dtype: int64

Validation complete.


Exact Data Duplicates checking

In [5]:

# Check exact duplicate rows in the original source data
print("Original rows:", len(df))
print("Exact duplicate rows:", df.duplicated().sum())

# Check repeated activities within the same patient journey
repeated_activities = (
    careflow_df.groupby(["case_id", "activity"])
    .size()
    .reset_index(name="occurrences")
    .query("occurrences > 1")
)

print(
    "Case-activity combinations repeated:",
    len(repeated_activities)
)

display(repeated_activities.head(10))


Original rows: 90965
Exact duplicate rows: 0
Case-activity combinations repeated: 0


,case_id,activity,occurrences


In [6]:

import pandas as pd

clean_df = careflow_df.copy()

initial_rows = len(clean_df)

# 1. Remove only exact duplicate records
clean_df = clean_df.drop_duplicates()

# 2. Remove rows missing essential process-mining fields
required_columns = [
    "case_id", "activity", "timestamp", "sequence"
]

clean_df = clean_df.dropna(subset=required_columns)

# 3. Standardize text fields
clean_df["case_id"] = clean_df["case_id"].astype(str).str.strip()
clean_df["activity"] = clean_df["activity"].astype(str).str.strip()

# 4. Ensure timestamps have valid datetime values
clean_df["timestamp"] = pd.to_datetime(
    clean_df["timestamp"], errors="coerce"
)

clean_df = clean_df.dropna(subset=["timestamp"])

# 5. Filter empty identifiers and activity names
clean_df = clean_df[
    clean_df["case_id"].ne("") &
    clean_df["activity"].ne("")
].copy()

# 6. Sort events chronologically within each case
clean_df = clean_df.sort_values(
    ["case_id", "timestamp", "sequence"],
    kind="stable"
).reset_index(drop=True)

# 7. Recalculate sequence numbers after filtering
clean_df["sequence"] = (
    clean_df.groupby("case_id").cumcount() + 1
)

print("Rows before cleaning:", initial_rows)
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", initial_rows - len(clean_df))
print("Unique cases:", clean_df["case_id"].nunique())
print("Missing required values:")
print(clean_df[required_columns].isna().sum())

display(clean_df.head(10))


Rows before cleaning: 90965
Rows after cleaning: 90965
Rows removed: 0
Unique cases: 16011
Missing required values:
case_id      0
activity     0
timestamp    0
sequence     0
dtype: int64


,case_id,activity,timestamp,sequence,triage_code,initial_zone,disposition_code,disposition_desc,age,gender
0,5240985,Left ED,2021-04-28 00:00:00,1,2.0,NaN,63,Left After Triage,8,M
1,5240985,Discharge,2021-04-28 00:00:00,2,2.0,NaN,63,Left After Triage,8,M
2,5240985,Triage,2021-04-28 01:30:00,3,2.0,NaN,63,Left After Triage,8,M
3,5240985,Registration,2021-04-28 01:45:00,4,2.0,NaN,63,Left After Triage,8,M
4,7214779,Triage,2021-05-21 11:17:00,1,3.0,YZ,63,Left After Triage,27,M
5,7214779,Registration,2021-05-21 11:27:00,2,3.0,YZ,63,Left After Triage,27,M
6,7214779,Left ED,2021-05-21 12:09:00,3,3.0,YZ,63,Left After Triage,27,M
7,7214779,Discharge,2021-05-21 12:09:00,4,3.0,YZ,63,Left After Triage,27,M
8,7268604,Ambulance Arrival,2021-04-11 00:00:00,1,2.0,Resus,7,Admit to reporting facility as inpatient to an...,27,F
9,7268604,Triage,2021-04-11 13:04:00,2,2.0,Resus,7,Admit to reporting facility as inpatient to an...,27,F


Final dataset download

In [7]:
from google.colab import files

files.download(
    "data/processed/careflow_event_log.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>